# Notebook 01 — Preprocesamiento y Análisis Exploratorio de Datos
## Proyecto 6: Clustering de Modos de Consumo Energético — BPC Estación de Bombeo

**Diplomado Ciencia de Datos | IDC Ingeniería de Confiabilidad S.A.S.**  
**Activo:** Bomba centrífuga con motor eléctrico de media tensión (~4.2 kV)  
**Dataset:** `Data_EBR_processed.csv` — 255,307 registros | 23 variables | Junio–Agosto 2024

---

### Propósito de este notebook

Antes de aplicar cualquier técnica de Machine Learning, es obligatorio entender y limpiar los datos. Este notebook cubre:

1. **Carga y diccionario de variables** — qué mide cada columna, en qué unidades, cuál es su rango físico esperado
2. **Evaluación de calidad del dato** — completitud, valores centinela, sensores congelados, gaps temporales
3. **Preprocesamiento justificado** — cada decisión de limpieza documenta el POR QUÉ, no solo el QUÉ
4. **Análisis exploratorio univariado y bivariado** — distribuciones, correlaciones, detección de patrones
5. **Selección de features para clustering** — cuáles variables entran al modelo y por qué

> **Principio de auditoría:** Todo cambio al dataset debe poder rastrearse. Ninguna fila se elimina sin registrar cuántas y por qué.

In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# Ejecuta esta celda UNA SOLA VEZ para instalar todo lo necesario
# ============================================================
import subprocess, sys

packages = [
    "pandas", "numpy", "scipy",
    "scikit-learn", "umap-learn",
    "matplotlib", "seaborn",
    "plotly", "kaleido",
    "openpyxl", "pyarrow", "fastparquet",
    "tqdm", "trimesh"
]

print("Instalando dependencias...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Listo. Puedes continuar con las siguientes celdas.")

---
## Sección 0 — Configuración del entorno

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3d4166',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#b0b0b0',
    'ytick.color': '#b0b0b0',
    'text.color': '#e0e0e0',
    'grid.color': '#2a2d3e',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
})
PALETTE = px.colors.qualitative.Bold

print('Entorno configurado correctamente.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

ModuleNotFoundError: No module named 'plotly'

---
## Sección 1 — Carga del Dataset y Diccionario de Variables

In [ ]:
# ------------------------------------------------------------
# AJUSTA ESTA RUTA si el CSV está en otra ubicación
# ------------------------------------------------------------
DATA_PATH = '../Data_EBR_processed (1).csv'
WEIGHTS_PATH = '../Pesos Ponderados - Proyecto6 (1).xlsx'

df_raw = pd.read_csv(DATA_PATH, parse_dates=['Fecha'])

print(f'Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Rango temporal : {df_raw["Fecha"].min()} → {df_raw["Fecha"].max()}')
print(f'Memoria         : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Diccionario de variables — referencia durante todo el proyecto
VARIABLE_DICT = {
    'Fecha'            : {'desc': 'Marca de tiempo del registro',                          'unit': '—',    'domain': 'temporal',  'rango_fisico': None},
    'Pres_Succ'        : {'desc': 'Presión en la línea de succión',                        'unit': 'PSI',  'domain': 'hidraulico', 'rango_fisico': (0, 400)},
    'Pres_Desc'        : {'desc': 'Presión en la línea de descarga',                       'unit': 'PSI',  'domain': 'hidraulico', 'rango_fisico': (0, 2000)},
    'Fluj_Desc'        : {'desc': 'Flujo volumétrico en la descarga',                      'unit': 'GPM',  'domain': 'hidraulico', 'rango_fisico': (0, 6000)},
    'Temp_BBA Acople'  : {'desc': 'Temperatura cojinete bomba — lado acople',              'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_BBA Oil-in'  : {'desc': 'Temperatura aceite lubricación — entrada',              'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_BBA Oil-out' : {'desc': 'Temperatura aceite lubricación — salida',               'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_BBA Carcasa' : {'desc': 'Temperatura de la carcasa de la bomba',                 'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_MTR Acople'  : {'desc': 'Temperatura cojinete motor — lado acople',              'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_MTR Libre'   : {'desc': 'Temperatura cojinete motor — lado libre',               'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 200)},
    'Temp_Devanado U'  : {'desc': 'Temperatura devanado motor — Fase U',                   'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 210)},
    'Temp_Devanado V'  : {'desc': 'Temperatura devanado motor — Fase V',                   'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 210)},
    'Temp_Devanado W'  : {'desc': 'Temperatura devanado motor — Fase W',                   'unit': '°C',   'domain': 'termico',   'rango_fisico': (20, 210)},
    'Corriente L1'     : {'desc': 'Corriente de línea L1 del motor',                       'unit': 'A',    'domain': 'electrico', 'rango_fisico': (0, 450)},
    'Corriente L2'     : {'desc': 'Corriente de línea L2 del motor',                       'unit': 'A',    'domain': 'electrico', 'rango_fisico': (0, 450)},
    'Corriente L3'     : {'desc': 'Corriente de línea L3 del motor',                       'unit': 'A',    'domain': 'electrico', 'rango_fisico': (0, 450)},
    'Voltaje L1-L2'    : {'desc': 'Voltaje de línea entre fases L1 y L2',                  'unit': 'kV',   'domain': 'electrico', 'rango_fisico': (3.8, 4.6)},
    'Voltaje L2-L3'    : {'desc': 'Voltaje de línea entre fases L2 y L3',                  'unit': 'kV',   'domain': 'electrico', 'rango_fisico': (3.8, 4.6)},
    'Voltaje L1-L3'    : {'desc': 'Voltaje de línea entre fases L1 y L3',                  'unit': 'kV',   'domain': 'electrico', 'rango_fisico': (3.8, 4.6)},
    'Potencia'         : {'desc': 'Potencia eléctrica activa consumida por el motor',       'unit': 'kW',   'domain': 'electrico', 'rango_fisico': (0, 3000)},
    'RPM'              : {'desc': 'Velocidad de rotación — variador de velocidad variable', 'unit': 'RPM',  'domain': 'mecanico',  'rango_fisico': (0, 4000)},
    'Pos_axial1'       : {'desc': 'Posición axial del eje — sensor 1',                      'unit': 'mm',   'domain': 'mecanico',  'rango_fisico': (-10, 10)},
    'Pos_axial2'       : {'desc': 'Posición axial del eje — sensor 2',                      'unit': 'mm',   'domain': 'mecanico',  'rango_fisico': (-10, 10)},
}

df_dict = pd.DataFrame(VARIABLE_DICT).T.reset_index().rename(columns={'index': 'Variable'})
print('Diccionario de variables:')
df_dict[['Variable', 'desc', 'unit', 'domain']]

---
## Sección 2 — Evaluación de Calidad del Dato

Antes de tocar los datos, documentamos su estado original. Esta sección es el **scorecard de calidad** que permite tomar decisiones de preprocesamiento fundamentadas.

In [ ]:
numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
N = len(df_raw)

# 2.1 Estadísticas descriptivas base
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
df_raw[numeric_cols].describe().round(2)

In [ ]:
# 2.2 Completitud y valores nulos
null_counts = df_raw[numeric_cols].isnull().sum()
null_pct    = (null_counts / N * 100).round(2)

df_quality = pd.DataFrame({
    'NaN count' : null_counts,
    'NaN %'     : null_pct,
    'Completitud %': (100 - null_pct).round(2)
})
print('=== COMPLETITUD ===')
print(df_quality.to_string())

In [ ]:
# 2.3 Detección de valores centinela (ceros que NO son medición real)
# En sistemas industriales, un cero en corriente/voltaje puede indicar sensor desconectado
zero_candidates = ['Corriente L1', 'Corriente L2', 'Corriente L3',
                   'Voltaje L1-L2', 'Voltaje L2-L3', 'Voltaje L1-L3',
                   'Potencia', 'Fluj_Desc']

print('=== CEROS POR VARIABLE ===')
for col in zero_candidates:
    n_zeros = (df_raw[col] == 0).sum()
    pct     = n_zeros / N * 100
    print(f'  {col:<22}: {n_zeros:>7,} ({pct:.2f}%)')

In [ ]:
# 2.4 Detección de sensor congelado (stuck sensor)
# Si una variable tiene el MISMO valor por >= 20 registros consecutivos
# mientras la bomba opera (RPM > 100), es sospechoso
STUCK_WINDOW = 20
print(f'=== SENSOR CONGELADO (ventana {STUCK_WINDOW} registros) ===')
for col in numeric_cols:
    consecutive_same = (df_raw[col].diff() == 0).astype(int)
    max_run = consecutive_same.groupby((consecutive_same != consecutive_same.shift()).cumsum()).cumsum().max()
    if max_run >= STUCK_WINDOW:
        print(f'  ALERTA: {col:<25} — racha máxima de {max_run} valores iguales consecutivos')

In [ ]:
# 2.5 Detección de gaps temporales
df_raw['Fecha'] = pd.to_datetime(df_raw['Fecha'], utc=True)
time_diffs = df_raw['Fecha'].diff().dropna().dt.total_seconds()
GAP_THRESHOLD = 60  # segundos

gaps = time_diffs[time_diffs > GAP_THRESHOLD]
print(f'=== GAPS TEMPORALES > {GAP_THRESHOLD}s ===')
print(f'  Gaps detectados : {len(gaps):,}')
print(f'  Gap máximo      : {gaps.max():.0f} s ({gaps.max()/3600:.2f} horas)')
print(f'  dt mediano      : {time_diffs.median():.1f} s')
print(f'  dt esperado     : ~14 s (según documento de entrega)')

# Top 5 gaps más grandes
print('\nTop 5 gaps mayores:')
top_gaps = gaps.nlargest(5)
for idx, val in top_gaps.items():
    print(f'  Índice {idx}: {val:.0f} s ({val/3600:.2f} h) — Fecha: {df_raw.loc[idx, "Fecha"]}')

In [ ]:
# 2.6 Varianza de voltajes (relevancia para clustering)
voltage_cols = ['Voltaje L1-L2', 'Voltaje L2-L3', 'Voltaje L1-L3']
print('=== VARIANZA NORMALIZADA DE VOLTAJES ===')
for col in voltage_cols:
    cv = df_raw[col].std() / df_raw[col].mean() * 100
    print(f'  {col}: CV = {cv:.4f}% — {'BAJA VARIANZA' if cv < 1 else 'varianza aceptable'}')
print('\nCONCLUSIÓN: Voltajes con CV < 1% aportan información mínima al clustering.')
print('Decisión pendiente: excluirlos o resumirlos según análisis de correlación.')

---
## Sección 3 — Preprocesamiento

### Principios que guían cada decisión:
1. **Trazabilidad**: todo registro eliminado se cuenta y justifica
2. **Dominio físico primero**: las leyes de la física son más confiables que cualquier umbral estadístico
3. **Conservadurismo**: en caso de duda, documentar y conservar antes de eliminar

### Secuencia de preprocesamiento:
```
Dataset crudo (255,307 filas)
  ↓ Paso 1: Rango físico → reemplazar por NaN
  ↓ Paso 2: Filtro de paros (RPM < 100 o Potencia = 0)
  ↓ Paso 3: Anomalías físicas imposibles (flags explícitos)
  ↓ Paso 4: Imputación residual (interpolación lineal)
  ↓ Paso 5: Verificación final
Dataset limpio (N filas)
```

In [ ]:
# Trabajamos sobre una COPIA — el raw nunca se modifica
df = df_raw.copy()
audit_log = []  # registro de auditoría: cada paso documenta cuántas filas se tocan

def log_step(step_name, n_before, n_after, reason):
    removed = n_before - n_after
    pct = removed / n_before * 100 if n_before > 0 else 0
    audit_log.append({
        'Paso': step_name,
        'Filas antes': n_before,
        'Filas después': n_after,
        'Eliminadas': removed,
        '% eliminado': round(pct, 3),
        'Justificación': reason
    })
    print(f'[{step_name}] {n_before:,} → {n_after:,} (-{removed:,} filas, {pct:.2f}%)')
    print(f'  Razón: {reason}')

print(f'Dataset inicial: {len(df):,} filas')

In [ ]:
# -------------------------------------------------------------------
# PASO 1: Rangos físicos — valores imposibles → NaN
# Justificación: un sensor no puede medir fuera de su rango físico.
# Si lo hace, es error de hardware o de comunicación, no un dato real.
# -------------------------------------------------------------------
n_before = len(df)
nan_introduced = 0

for var, meta in VARIABLE_DICT.items():
    if meta['rango_fisico'] is None or var not in df.columns:
        continue
    lo, hi = meta['rango_fisico']
    mask_out = (df[var] < lo) | (df[var] > hi)
    n_out = mask_out.sum()
    if n_out > 0:
        df.loc[mask_out, var] = np.nan
        nan_introduced += n_out
        print(f'  {var:<22}: {n_out:,} valores fuera de [{lo}, {hi}] → NaN')

print(f'\nTotal NaN introducidos por rangos físicos: {nan_introduced:,}')
log_step('1_rangos_fisicos', n_before, n_before, f'{nan_introduced} valores → NaN (filas no eliminadas aún)')

In [ ]:
# -------------------------------------------------------------------
# PASO 2: Eliminación de períodos de paro
#
# CRITERIO 1: RPM < 100
#   - La bomba opera con variador de velocidad variable (VSD).
#   - Por debajo de 100 RPM no hay hidrodinámica estable.
#   - Incluir estos puntos contaminaría los clusters de operación normal.
#
# CRITERIO 2: Potencia = 0
#   - Motor apagado o en estado de espera.
#   - Físicamente incompatible con operación productiva.
#
# DECISIÓN: Estos puntos NO se tratan como un modo operativo independiente.
# El objetivo del proyecto es identificar MODOS DE CONSUMO ENERGÉTICO,
# y un paro no consume energía de forma hidráulicamente significativa.
# -------------------------------------------------------------------
RPM_MIN  = 100
POT_MIN  = 0   # estrictamente mayor que cero

n_before = len(df)
mask_paro = (df['RPM'] < RPM_MIN) | (df['Potencia'] <= POT_MIN)
n_paros = mask_paro.sum()

# Detalle de cuántos caen en cada categoría
n_rpm_bajo   = (df['RPM'] < RPM_MIN).sum()
n_pot_cero   = (df['Potencia'] <= POT_MIN).sum()
n_ambos      = ((df['RPM'] < RPM_MIN) & (df['Potencia'] <= POT_MIN)).sum()

print(f'  RPM < {RPM_MIN}         : {n_rpm_bajo:,} registros')
print(f'  Potencia = 0        : {n_pot_cero:,} registros')
print(f'  Ambas condiciones   : {n_ambos:,} registros (overlap)')
print(f'  Total a eliminar    : {n_paros:,} registros ({n_paros/n_before*100:.2f}%)')

df = df[~mask_paro].reset_index(drop=True)
log_step('2_filtro_paros', n_before, len(df),
         f'RPM < {RPM_MIN} OR Potencia = 0 → paros/arranques excluidos del análisis de eficiencia')

In [ ]:
# -------------------------------------------------------------------
# PASO 3: Detección de anomalías físicas imposibles
#
# Observación del cliente: existen puntos con combinaciones físicamente
# inconsistentes. Las analizamos explícitamente.
#
# CASO A: Potencia = 0 PERO Flujo > 0 Y Presión descarga > umbral
#   - Un motor que no consume energía NO puede mover fluido.
#   - Hipótesis: retardo de muestreo entre sensores (el flujómetro
#     aún registra inercia del fluido segundos después del paro),
#     o error de comunicación del sensor de potencia.
#   - Acción: flag y eliminación (ya cubiertos por Paso 2, pero
#     los documentamos aquí para auditoría).
#
# CASO B: Potencia > 0 Y Flujo > 0 PERO Presión descarga < 50 PSI
#   - La bomba consume energía y hay flujo, pero la presión es anormalmente
#     baja para este tipo de sistema (presión normal: 800-1600 PSI).
#   - Hipótesis: sensor de presión de descarga con falla o lectura errónea.
#   - Acción: NaN en Pres_Desc para esos registros, mantener el punto.
#
# CASO C: RPM > 500 PERO Fluj_Desc = 0
#   - Motor girando pero sin flujo. Puede indicar cavitación severa o
#     válvula de descarga completamente cerrada.
#   - Acción: flag para análisis pero NO eliminar — puede ser un modo
#     operativo real (recirculación interna).
# -------------------------------------------------------------------

PRES_DESC_UMBRAL = 50   # PSI
FLUJO_UMBRAL     = 10   # GPM

# Caso A (ya eliminados, verificación)
caso_a = ((df['Potencia'] <= 0) & (df['Fluj_Desc'] > FLUJO_UMBRAL)).sum()
print(f'Caso A (Pot=0, Flujo>0): {caso_a:,} registros restantes → OK (cubiertos en Paso 2)')

# Caso B
mask_b = (df['Potencia'] > 0) & (df['Fluj_Desc'] > FLUJO_UMBRAL) & (df['Pres_Desc'] < PRES_DESC_UMBRAL)
n_caso_b = mask_b.sum()
print(f'Caso B (Pot>0, Flujo>0, Pres_Desc<{PRES_DESC_UMBRAL}): {n_caso_b:,} registros')
if n_caso_b > 0:
    df.loc[mask_b, 'Pres_Desc'] = np.nan
    print(f'  → Pres_Desc marcada como NaN en esos {n_caso_b:,} registros')

# Caso C
mask_c = (df['RPM'] > 500) & (df['Fluj_Desc'] <= FLUJO_UMBRAL)
n_caso_c = mask_c.sum()
df['flag_flujo_cero_operando'] = mask_c.astype(int)
print(f'Caso C (RPM>500, Flujo≈0): {n_caso_c:,} registros → flaggeados, NO eliminados')
print(f'  Interpretación: posible recirculación, cavitación, o válvula cerrada.')

In [ ]:
# -------------------------------------------------------------------
# PASO 4: Imputación de NaN residuales
#
# Estrategia por tipo de variable:
# - Temperaturas: interpolación lineal (variable lenta y continua)
# - Presiones/Flujo: interpolación lineal (si el gap < 10 registros)
# - Corrientes: forward-fill (muestreo discreto)
# - Voltajes: forward-fill (cuasi-constante)
# -------------------------------------------------------------------
temp_cols      = [c for c in df.columns if 'Temp_' in c]
hidro_cols     = ['Pres_Succ', 'Pres_Desc', 'Fluj_Desc']
corriente_cols = ['Corriente L1', 'Corriente L2', 'Corriente L3']
voltaje_cols   = ['Voltaje L1-L2', 'Voltaje L2-L3', 'Voltaje L1-L3']

MAX_INTERP_GAP = 10  # máximo de NaN consecutivos a interpolar

for col in temp_cols + hidro_cols:
    n_nan = df[col].isnull().sum()
    if n_nan > 0:
        df[col] = df[col].interpolate(method='linear', limit=MAX_INTERP_GAP, limit_direction='both')
        print(f'  {col:<25}: {n_nan:,} NaN → interpolación lineal (max gap={MAX_INTERP_GAP})')

for col in corriente_cols + voltaje_cols:
    n_nan = df[col].isnull().sum()
    if n_nan > 0:
        df[col] = df[col].ffill().bfill()
        print(f'  {col:<25}: {n_nan:,} NaN → forward-fill / backward-fill')

# Verificación final
nan_residual = df[numeric_cols].isnull().sum().sum()
print(f'\nNaN residuales después de imputación: {nan_residual}')
log_step('4_imputacion_nan', len(df), len(df), 'Interpolación y forward-fill según tipo de variable')

In [ ]:
# -------------------------------------------------------------------
# PASO 5: Resumen del preprocesamiento
# -------------------------------------------------------------------
df_audit = pd.DataFrame(audit_log)
print('=== LOG DE AUDITORÍA DEL PREPROCESAMIENTO ===')
print(df_audit.to_string(index=False))
print(f'\nDataset LIMPIO: {len(df):,} filas × {df.shape[1]} columnas')
print(f'Reducción total: {len(df_raw) - len(df):,} filas ({(len(df_raw) - len(df))/len(df_raw)*100:.2f}%)')

---
## Sección 4 — Análisis Exploratorio de Datos (EDA)

In [ ]:
# 4.1 Series de tiempo — variables clave de consumo energético
KEY_VARS = ['RPM', 'Potencia', 'Fluj_Desc', 'Pres_Desc', 'Pres_Succ']
LABELS   = ['RPM [RPM]', 'Potencia [kW]', 'Flujo Desc [GPM]', 'Pres. Desc [PSI]', 'Pres. Succ [PSI]']

fig = make_subplots(rows=len(KEY_VARS), cols=1, shared_xaxes=True,
                    subplot_titles=LABELS,
                    vertical_spacing=0.04)

colors = ['#4FC3F7', '#FF8A65', '#81C784', '#CE93D8', '#FFD54F']
for i, (col, label, color) in enumerate(zip(KEY_VARS, LABELS, colors), start=1):
    fig.add_trace(go.Scatter(
        x=df['Fecha'], y=df[col],
        mode='lines', name=label,
        line=dict(width=0.7, color=color),
        opacity=0.85
    ), row=i, col=1)

fig.update_layout(
    height=900, title_text='Series de Tiempo — Variables de Consumo Energético (post-preprocesamiento)',
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
    font=dict(color='#e0e0e0'), showlegend=False
)
fig.update_xaxes(gridcolor='#2a2d3e')
fig.update_yaxes(gridcolor='#2a2d3e')
fig.write_html('../outputs/html/series_tiempo_energia.html')
fig.show()

In [ ]:
# 4.2 Distribuciones — histogramas con KDE para variables clave
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Distribuciones — Variables Principales (operación estable)', fontsize=13, y=1.01)

vars_hist = ['RPM', 'Potencia', 'Fluj_Desc', 'Pres_Desc', 'Pres_Succ', 'Corriente L1']
units     = ['RPM', 'kW', 'GPM', 'PSI', 'PSI', 'A']
pal_hist  = ['#4FC3F7', '#FF8A65', '#81C784', '#CE93D8', '#FFD54F', '#80CBC4']

for ax, col, unit, color in zip(axes.flat, vars_hist, units, pal_hist):
    data = df[col].dropna()
    ax.hist(data, bins=80, color=color, alpha=0.75, edgecolor='none')
    ax.axvline(data.mean(),   color='white',  lw=1.5, ls='--', label=f'Media: {data.mean():.1f}')
    ax.axvline(data.median(), color='yellow', lw=1.0, ls=':',  label=f'Mediana: {data.median():.1f}')
    ax.set_title(f'{col} [{unit}]')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/distribuciones_variables_clave.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 Multimodalidad en RPM — clave para el clustering
# La variación de RPM es la fuente principal de variabilidad del sistema.
# Un histograma detallado revela si hay modos de velocidad discretos.
fig_rpm = px.histogram(
    df, x='RPM', nbins=150,
    title='Distribución de RPM — ¿Hay modos de velocidad discretos?',
    color_discrete_sequence=['#4FC3F7'],
    labels={'RPM': 'Velocidad de Rotación [RPM]'}
)
fig_rpm.update_layout(
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
    font=dict(color='#e0e0e0')
)
fig_rpm.write_html('../outputs/html/distribucion_rpm.html')
fig_rpm.show()
print('OBSERVAR: ¿el histograma muestra picos discretos (setpoints del VSD)?')
print('Si hay picos: los clusters de velocidad tendrán fronteras naturales.')
print('Si es continuo: el VSD opera en modo analógico continuo.')

In [ ]:
# 4.4 Matriz de correlación — base para la selección de features
# Solo variables numéricas relevantes (excluimos flags)
corr_cols = [c for c in numeric_cols if 'flag_' not in c]
corr_matrix = df[corr_cols].corr(method='spearman')  # Spearman: robusto a no-linealidad

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#0f1117',
    annot_kws={'size': 6}, ax=ax
)
ax.set_title('Matriz de Correlación de Spearman — Dataset Preprocesado', pad=15)
plt.tight_layout()
plt.savefig('../outputs/figures/matriz_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

# Pares altamente correlacionados (|r| > 0.85)
print('\nPARES CON |r| > 0.85 (candidatos a reducción):')
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.85:
            high_corr.append({'Var1': corr_matrix.columns[i], 'Var2': corr_matrix.columns[j], 'r': round(r, 3)})
            print(f'  {corr_matrix.columns[i]:<25} ↔ {corr_matrix.columns[j]:<25} : r = {r:.3f}')

In [ ]:
# 4.5 Scatter plots bivariados — relaciones físicas clave
# Estas gráficas revelan si hay clusters VISIBLES antes del ML formal

scatter_pairs = [
    ('RPM', 'Potencia',  'Velocidad vs Potencia — curva de carga del sistema'),
    ('RPM', 'Fluj_Desc', 'Velocidad vs Flujo — curva de bomba'),
    ('Fluj_Desc', 'Pres_Desc', 'Flujo vs Presión de Descarga — curva H-Q'),
    ('Fluj_Desc', 'Potencia',  'Flujo vs Potencia — curva de potencia'),
]

for x_col, y_col, title in scatter_pairs:
    # Muestra aleatoria para velocidad de renderizado
    sample = df[[x_col, y_col, 'RPM']].dropna().sample(min(15000, len(df)), random_state=42)
    fig = px.scatter(
        sample, x=x_col, y=y_col, color='RPM',
        color_continuous_scale='Viridis',
        title=title, opacity=0.4,
        labels={x_col: f'{x_col} [{VARIABLE_DICT.get(x_col, {}).get("unit", "")}]',
                y_col: f'{y_col} [{VARIABLE_DICT.get(y_col, {}).get("unit", "")}]'}
    )
    fig.update_traces(marker=dict(size=2))
    fig.update_layout(paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27', font=dict(color='#e0e0e0'))
    fname = f'../outputs/html/scatter_{x_col}_{y_col}.html'.replace(' ', '_')
    fig.write_html(fname)
    fig.show()

---
## Sección 5 — Selección de Features para Clustering

### Criterios de inclusión/exclusión:

| Variable | Decisión | Justificación |
|----------|----------|---------------|
| RPM | **INCLUIR** | Principal fuente de variabilidad (VSD). Determina el régimen operativo. |
| Potencia | **INCLUIR** | KPI principal de consumo energético. |
| Fluj_Desc | **INCLUIR** | Define la carga hidráulica. |
| Pres_Desc | **INCLUIR** | Define la carga de presión del sistema. |
| Pres_Succ | **INCLUIR** | Diferencial de presión (Head total = Pres_Desc - Pres_Succ). |
| Corriente L1/L2/L3 | **INCLUIR 1** | Las tres fases están altamente correlacionadas → promedio. |
| Temp_BBA Acople | **INCLUIR** | Indicador de carga mecánica y fricción. |
| Temp_MTR Acople | **INCLUIR** | Indicador de carga eléctrica del motor. |
| Temp_Devanado U/V/W | **INCLUIR 1** | Altamente correlacionadas → promedio. |
| Voltaje L1-L2/L2-L3/L1-L3 | **EXCLUIR** | CV < 1% → varianza mínima, aportan ruido al clustering. |
| Pos_axial1/2 | **EXCLUIR** | Desplazamiento axial: mecánico, no energético. Fuera del scope. |
| Temp_BBA Oil-in/out | **EXCLUIR** | Alta correlación con Temp_BBA Acople (r > 0.9). |
| Temp_BBA Carcasa | **EXCLUIR** | Alta correlación con Temp_MTR Libre (r > 0.88). |

In [ ]:
# Variables sintéticas derivadas
# Aportan información física directa que mejora la separación entre clusters

# Corriente promedio de las 3 fases
df['Corriente_prom'] = df[['Corriente L1', 'Corriente L2', 'Corriente L3']].mean(axis=1)

# Temperatura promedio de devanados (térmica eléctrica)
df['Temp_devanado_prom'] = df[['Temp_Devanado U', 'Temp_Devanado V', 'Temp_Devanado W']].mean(axis=1)

# Head diferencial total (presión diferencial bomba)
# TDH [PSI] = Pres_Desc - Pres_Succ (simplificación: misma cota, igual diámetro)
df['TDH_PSI'] = df['Pres_Desc'] - df['Pres_Succ']

# Potencia específica (proxy de eficiencia relativa: kW por GPM)
# Menor valor → más eficiente para ese flujo
df['Potencia_especifica'] = df['Potencia'] / (df['Fluj_Desc'] + 1e-6)  # +1e-6 evita división por cero

print('Variables sintéticas creadas:')
print(f'  Corriente_prom    : media de L1, L2, L3')
print(f'  Temp_devanado_prom: media de Devanado U, V, W')
print(f'  TDH_PSI           : Pres_Desc - Pres_Succ (Head Total Dinámico)')
print(f'  Potencia_especifica: Potencia / Flujo (kW/GPM — proxy de eficiencia)')

In [ ]:
# Features finales para clustering
FEATURES_CLUSTERING = [
    'RPM',
    'Potencia',
    'Fluj_Desc',
    'Pres_Desc',
    'TDH_PSI',
    'Corriente_prom',
    'Temp_BBA Acople',
    'Temp_MTR Acople',
    'Temp_devanado_prom',
]

df_features = df[FEATURES_CLUSTERING].dropna().copy()
print(f'Dataset de features para clustering: {df_features.shape[0]:,} filas × {df_features.shape[1]} features')
print(f'Features seleccionadas: {FEATURES_CLUSTERING}')

# Estadísticas del espacio de features
df_features.describe().round(2)

In [ ]:
# Exportar dataset limpio para notebooks siguientes
df.to_parquet('../outputs/df_clean.parquet', index=False)
df_features.to_parquet('../outputs/df_features.parquet', index=False)
print(f'Dataset limpio guardado: ../outputs/df_clean.parquet ({len(df):,} filas)')
print(f'Features guardadas: ../outputs/df_features.parquet ({len(df_features):,} filas)')

---
## Resumen del Notebook 01

| Etapa | Resultado |
|-------|-----------|
| Dataset original | 255,307 filas × 23 columnas |
| Después de preprocesamiento | Ver log de auditoría arriba |
| Features para clustering | 9 variables (5 originales + 4 sintéticas/reducidas) |
| Anomalías físicas documentadas | Casos A, B, C |

**Siguiente paso:** `02_clustering.ipynb` — aplicar y evaluar algoritmos de clustering sobre el espacio de 9 features.